In [20]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


In [21]:
file_dict = {
    'mouse1': {
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251217_225054',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1&oldV1_natima_251219_201746'
    },
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [22]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [19]:
# 处理合并数据的后处理：为每个clique生成neuron_inf和gt_detect_array，并拆分到各个日期
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd

mouse_name = 'mouse2'
combined_output_base = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}'
dates_list = [1214, 1215, 1216, 1217, 1218, 1219]  # mouse1的日期列表，按顺序

# 构建cliques
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

print(f"\n{'='*60}")
print(f"读取并合并 {mouse_name} 的所有日期数据")
print(f"{'='*60}")

all_recordings_list = []
channel_list = None
# 记录每个日期的采样点数（在resample之前）
date_num_samples = {}  # {date: num_samples}

for date in dates_list:
    data_path = file_dict[mouse_name][date]
    
    # 获取该数据路径下的所有rhd文件
    file_list_path = Path(data_path)
    rhd_files = list(file_list_path.glob("*.rhd"))
    file_list = sorted(rhd_files)
    

    
    # 读取并合并该date的所有rhd文件
    recording_raw_list = []
    for file in file_list:
        recording_raw_list.append(se.read_intan(file, stream_id='0'))
    
    if len(recording_raw_list) > 0:
        date_recording = concatenate_recordings(recording_list=recording_raw_list)
        
        # 检测通道类型并选择对应的channel_list
        available_channels = date_recording.get_channel_ids()
        if 'A-127' in available_channels:
            channel_list = channel_list_A
        elif 'B-127' in available_channels:
            channel_list = channel_list_B
        else:
            print(f"警告: 未找到A-127或B-127通道，跳过此日期")
            continue
        
        # 选择通道
        date_recording = date_recording.select_channels(channel_list)
        
        # 统一将B开头的channel重命名为A开头
        channel_ids = date_recording.get_channel_ids()
        new_channel_ids = []
        renamed_count = 0
        for ch_id in channel_ids:
            if isinstance(ch_id, str) and ch_id.startswith('B-'):
                new_ch_id = 'A-' + ch_id[2:]
                new_channel_ids.append(new_ch_id)
                renamed_count += 1
            else:
                new_channel_ids.append(ch_id)
        
        if renamed_count > 0:
            date_recording = date_recording.rename_channels(new_channel_ids)
        
        # 记录该日期的原始采样点数（在resample之前）
        # 注意：这里记录的是原始采样点数，后续会resample到10000Hz
        # 但我们需要知道resample后的采样点数
        all_recordings_list.append(date_recording)


recording_combined = concatenate_recordings(recording_list=all_recordings_list)

print("\n开始预处理...")
recording_raw = spre.unsigned_to_signed(recording_combined)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

# 计算每个日期在resample后的采样点数
# 需要在resample之前记录原始采样点数和采样率，然后计算resample后的采样点数
target_sampling_frequency = 10000.0
for i, date in enumerate(dates_list):
    if i < len(all_recordings_list):
        date_recording = all_recordings_list[i]
        original_sampling_freq = date_recording.get_sampling_frequency()
        original_num_samples = date_recording.get_num_samples()
        # resample后的采样点数 = 原始采样点数 * (目标采样率 / 原始采样率)
        resampled_num_samples = int(original_num_samples * (target_sampling_frequency / original_sampling_freq))
        date_num_samples[date] = resampled_num_samples
        print(f"Date {date}: 原始采样点数 = {original_num_samples}, 原始采样率 = {original_sampling_freq:.1f} Hz, resample后采样点数 = {resampled_num_samples}")

# 计算每个segment的采样点范围（基于date_num_samples）
segment_sample_ranges_by_date = {}  # {segment_idx: (start_sample, end_sample)}
cumulative_samples = 0
for segment_idx, date in enumerate(dates_list):
    if date in date_num_samples:
        num_samples = date_num_samples[date]
        segment_sample_ranges_by_date[segment_idx] = (cumulative_samples, cumulative_samples + num_samples)
        cumulative_samples += num_samples
        print(f"Segment {segment_idx} (Date {date}): 采样点范围 = [{segment_sample_ranges_by_date[segment_idx][0]}, {segment_sample_ranges_by_date[segment_idx][1]})")

# 对每个clique进行处理
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理 Clique {clique_id}")
    print(f"{'='*60}")
    
    clique_output_folder = f'{combined_output_base}/clique_{clique_id}'
    phy_folder = f'{clique_output_folder}/phy_folder_for_kilosort'
    
    # 读取sorting结果
    sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise"])
    print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units")
    
    recording_combined_clique = get_recording_clique(recording_f, clique)
    analyzer_curated_phy = si.create_sorting_analyzer(
        sorting=sorting_curated_phy, 
        recording=recording_combined_clique, 
        format='binary_folder',
        folder=clique_output_folder + '/analyzer_curated_temp',
        n_jobs=20, verbose = False
    )
    
    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "templates",
        "unit_locations",
        "template_similarity"
    ]
    
    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "template_similarity": {"method": "cosine_similarity"}
    }
    
    analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    

    
    # 获取neuron信息（与concat_post中相同）
    templates_ext = analyzer_curated_phy.get_extension("templates")
    templates_dense = templates_ext.data["average"]
    sparsity = analyzer_curated_phy.sparsity
    unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations = unit_locations_ext.get_data()
    channel_locations = analyzer_curated_phy.get_channel_locations()
    
    # 处理merge逻辑（与concat_post相同）
    if unit_locations.shape[1] >= 2:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations[:, :2], 
            unit_locations[:, :2], 
            metric="euclidean"
        )
    else:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations, 
            unit_locations, 
            metric="euclidean"
        )
    
    template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
    template_similarity = template_similarity_ext.get_data()
    
    distance_threshold = 10.0
    similarity_threshold = 0.95
    num_units = len(analyzer_curated_phy.unit_ids)
    pair_mask = np.zeros((num_units, num_units), dtype=bool)
    
    for i in range(num_units):
        for j in range(i + 1, num_units):
            if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
                pair_mask[i, j] = True
                pair_mask[j, i] = True
    
    n_components, labels = connected_components(
        csgraph=pair_mask, 
        directed=False, 
        return_labels=True
    )
    
    merge_unit_groups = []
    unit_ids_list = analyzer_curated_phy.unit_ids
    for component_id in range(n_components):
        unit_indices = np.where(labels == component_id)[0]
        if len(unit_indices) > 1:
            group = [unit_ids_list[i] for i in unit_indices]
            merge_unit_groups.append(group)
    
    # 应用merge（如果有需要merge的units）
    if len(merge_unit_groups) > 0:
        analyzer_merged = analyzer_curated_phy.merge_units(
            merge_unit_groups=merge_unit_groups,
            censor_ms=0.3,
            merging_mode="hard",
            new_id_strategy="append",
            format='binary_folder',
            folder=clique_output_folder + '/analyzer_merged',
            verbose=True,
            n_jobs=20
        )
        
        analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
        
        templates_ext_final = analyzer_merged.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_merged.sparsity
        unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_merged.get_channel_locations()
        sorting_final = analyzer_merged.sorting
        unit_ids_list_final = analyzer_merged.unit_ids
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_merged.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_merged, 
            peak_sign="neg",
            outputs="id"
        )
    else:
        # 不需要merge，使用原始结果
        templates_ext_final = analyzer_curated_phy.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_curated_phy.sparsity
        unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_curated_phy.get_channel_locations()
        sorting_final = analyzer_curated_phy.sorting
        unit_ids_list_final = unit_ids_list
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_curated_phy, 
            peak_sign="neg",
            outputs="id"
        )
    
    # 获取channel_ids（用于将通道索引转换为通道名称）
    # 使用analyzer的recording来获取channel_ids
    if len(merge_unit_groups) > 0:
        channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
    else:
        channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())
    
    # 计算每个unit的channel_id（template中值不为0的通道）
    channel_ids_dict = {}  # {unit_id: [channel_id1, channel_id2, ...]}
    for idx, unit_id in enumerate(unit_ids_list_final):
        unit_index = sorting_final.id_to_index(unit_id)
        template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
        
        # 找到template中值不为0的通道
        # 检查每个通道是否有非零值（在整个时间窗口内）
        non_zero_channels = []
        for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
            if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
                # 将通道索引转换为通道名称（格式和extremum_channel一样）
                channel_name = str(channel_ids_list[ch_idx])
                non_zero_channels.append(channel_name)
        
        channel_ids_dict[unit_id] = non_zero_channels
    
    # 计算channel_snr（每个unit的各个channel的SNR）
    print("计算channel_snr...")
    recording_combined_clique = get_recording_clique(recording_f, clique)
    n_channels = recording_combined_clique.get_num_channels()
    sampling_frequency = recording_combined_clique.get_sampling_frequency()
    
    # 1. 计算noise_std（使用和detect_spike相同的方法）
    # 读取一小段数据来计算noise_std（使用前10秒的数据）
    duration_samples = int(10 * sampling_frequency)  # 10秒
    max_samples = recording_combined_clique.get_num_samples()
    actual_samples = min(duration_samples, max_samples)
    traces = recording_combined_clique.get_traces(start_frame=0, end_frame=actual_samples)  # (n_timepoints, n_channels)

    noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

    all_spike_times = []
    all_spike_unit_ids = []
    for unit_id in unit_ids_list_final:
        spike_train = sorting_final.get_unit_spike_train(unit_id)
        all_spike_times.extend(spike_train.tolist())
        all_spike_unit_ids.extend([unit_id] * len(spike_train))
    
    n_spikes_total = len(all_spike_times)
    n_spikes_sample = min(1000, n_spikes_total)
    if n_spikes_sample > 0:
        random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
        sampled_spike_times = [all_spike_times[i] for i in random_indices]
        sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
    else:
        sampled_spike_times = []
        sampled_spike_unit_ids = []
    
    # 4. 提取这些spike的waveform并计算每个channel的负值amplitude
    left_sample = 10
    right_sample = 20
    window_size = left_sample + right_sample
    
    channel_snr_dict = {} 
    
    for unit_id in unit_ids_list_final:
        channel_snr_dict[unit_id] = {}
        unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
        
        if len(unit_spike_times) == 0:
            unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
            if len(unit_spike_times) > 1000:
                unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
        
        unit_waveforms = []  # List of (n_channels, window_size)
        valid_spike_times = []
        
        for spike_time in unit_spike_times:
            start = spike_time - left_sample
            end = spike_time + right_sample

            waveform = recording_combined_clique.get_traces(start_frame=start, end_frame=end)  # (n_channels, window_size)
            unit_waveforms.append(waveform)
            valid_spike_times.append(spike_time)
        
        if len(unit_waveforms) == 0:
            continue
        
        unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_channels, window_size)
        
        spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
        
        channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
        channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
        
        # 只保存 channel_ids_dict[unit_id] 中列出的通道的 SNR
        channel_ids_list = list(recording_combined_clique.get_channel_ids())
        unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
        
        for ch_idx, snr_value in enumerate(channel_snr):
            channel_id = str(channel_ids_list[ch_idx])
            # 只保存 channel_ids_dict 中列出的通道
            if channel_id in unit_channel_ids:
                channel_snr_dict[unit_id][channel_id] = float(snr_value)
    
    print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units")
    
    # 生成整体的neuron_inf
    neuron_inf = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        neuron_inf[unit_id] = {
            'location_x': float(unit_locations_final[idx, 0]),
            'location_y': float(unit_locations_final[idx, 1]),
            'position_waveform': position_waveforms_final[idx],
            'extremum_channel': extremum_channels_final[unit_id],
            'channel_id': channel_ids_dict[unit_id],
            'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
        }
    
    # 保存整体的neuron_inf
    with open(clique_output_folder + '/neuron_inf.pickle', 'wb') as f:
        pickle.dump(neuron_inf, f)
    
    # 生成整体的gt_detect_array（使用合并数据的时间）
    recording_combined_clique = get_recording_clique(recording_f, clique)
    sampling_frequency = recording_combined_clique.get_sampling_frequency()
    # 使用之前计算的segment_sample_ranges_by_date（基于日期划分）
    segment_sample_ranges = segment_sample_ranges_by_date.copy()
    
    spike_vector_final = sorting_final.to_spike_vector()
    gt_detect_data_all = []
    
    for spike in spike_vector_final:
        unit_index = spike['unit_index']
        unit_id = sorting_final.unit_ids[unit_index]
        sample_index = spike['sample_index']  # 全局采样点索引（跨所有日期）
        
        # 根据sample_index确定它属于哪个segment（日期）
        # sample_index是全局的，需要根据segment_sample_ranges来确定属于哪个segment
        segment_index = None
        for seg_idx, (start_sample, end_sample) in segment_sample_ranges.items():
            if start_sample <= sample_index < end_sample:
                segment_index = seg_idx
                break
        
        if segment_index is None:
            # 如果无法确定segment，跳过（理论上不应该发生）
            continue
        
        # 计算segment内的相对采样点索引（精确到每个采样点）
        segment_start_sample, segment_end_sample = segment_sample_ranges[segment_index]
        relative_sample_index = sample_index - segment_start_sample
        
        time_seconds = relative_sample_index
        
        extremum_channel = extremum_channels_final[unit_id]
        
        gt_detect_data_all.append({
            'time': time_seconds,
            'unit_id': unit_id,
            'extremum_channel': str(extremum_channel),
            'segment_index': segment_index  # 保存segment_index用于精确筛选
        })
    
    gt_detect_array_all = pd.DataFrame(gt_detect_data_all)
    
    gt_detect_array_all.to_csv(clique_output_folder + '/gt_detect_array.csv', index=False)
    

    for segment_idx, date in enumerate(dates_list):
        print(f"\n处理日期 {date} (Segment {segment_idx})...")
        
        # 使用segment_index进行精确筛选，而不是时间范围
        # 这样可以确保精确到每个采样点，避免浮点数精度问题
        mask = (gt_detect_array_all['segment_index'] == segment_idx)
        
        spikes_in_segment = gt_detect_array_all[mask].copy()
        
        # time已经是segment内的相对时间（从0开始），精确到采样点
        # 删除segment_index列（因为已经筛选完成，不再需要）
        if 'segment_index' in spikes_in_segment.columns:
            spikes_in_segment = spikes_in_segment.drop(columns=['segment_index'])
        
        # 筛选neuron：计算每个neuron的firing rate，如果 < 1 Hz则删除
        # 使用采样点数计算：firing_rate = spike_count * sampling_frequency / num_samples
        neurons_in_date = spikes_in_segment['unit_id'].unique()
        firing_rate_threshold = 0.5  # Hz
        valid_neurons = []
        removed_spikes_count = 0
        
        # 获取该segment的采样点数（用于计算firing rate）
        # 使用date_num_samples获取该日期在resample后的采样点数
        if date not in date_num_samples:
            print(f"  警告: 日期 {date} 的采样点数未找到，跳过")
            continue
        segment_num_samples = date_num_samples[date]
        
        for unit_id in neurons_in_date:
            if unit_id not in neuron_inf:
                # 如果neuron不在neuron_inf中，跳过
                continue
            
            # 计算该neuron在该segment中的spike数量
            neuron_spikes = spikes_in_segment[spikes_in_segment['unit_id'] == unit_id]
            spike_count = len(neuron_spikes)
            
            # 计算firing rate (spikes per second)，使用采样点数：spike_count * sampling_frequency / num_samples
            firing_rate = (spike_count * sampling_frequency) / segment_num_samples if segment_num_samples > 0 else 0
            
            if firing_rate >= firing_rate_threshold:
                valid_neurons.append(unit_id)
            else:
                # 删除该neuron的spikes
                removed_spikes_count += spike_count
                spikes_in_segment = spikes_in_segment[spikes_in_segment['unit_id'] != unit_id]
        
        # 只保留有效的neurons
        neuron_inf_date = {unit_id: neuron_inf[unit_id] for unit_id in valid_neurons}
        

        date_output_folder = f'{clique_output_folder}/date_{date}'
        os.makedirs(date_output_folder, exist_ok=True)
        
        with open(date_output_folder + '/neuron_inf.pickle', 'wb') as f:
            pickle.dump(neuron_inf_date, f)
        
        spikes_in_segment.to_csv(date_output_folder + '/gt_detect_array.csv', index=False)
        
        print(f"  日期 {date} 结果已保存到: {date_output_folder}")
        print(f"    - Neurons: {len(neuron_inf_date)}")
        print(f"    - Spikes: {len(spikes_in_segment)}")

print("\n所有clique处理完成！")


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks

读取并合并 mouse2 的所有日期数据

开始预处理...
Date 1214: 原始采样点数 = 31746944, 原始采样率 = 20000.0 Hz, resample后采样点数 = 15873472
Date 1215: 原始采样点数 = 25614464, 原始采样率 = 20000.0 Hz, resample后采样点数 = 12807232
Date 1216: 原始采样点数 = 25874944, 原始采样率 = 20000.0 Hz, resample后采样点数 = 12937472
Date 1217: 原始采样点数 = 43420416, 原始采样率 = 20000.0 Hz, resample后采样点数 = 21710208
Date 1218: 原始采样点数 = 55462528, 原始采样率 = 20000.0 Hz, resample后采样点数 = 27731264
Date 1219: 原始采样点数 = 34041344, 原始采样率 = 20000.0 Hz, resample后采样点数 = 17020672
Segment 0 (Date 1214): 采样点范围 = [0, 15873472)
Segment 1 (Date 1215): 采样点范围 = [15873472, 28680704)
Segment 2 (Date 1216): 采样点范围 = [28680704, 41618176)
Segment 3 (Date 1217): 采样点范围 = [41618176, 63328384)
Segment 4 (Date 1218): 采样点范围 = [63328384, 91059648)
Segment 5 (Date 1219): 采样点

estimate_sparsity (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

compute_waveforms (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

计算channel_snr...
完成channel_snr计算，共处理33个units

处理日期 1214 (Segment 0)...
  日期 1214 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1214
    - Neurons: 17
    - Spikes: 340832

处理日期 1215 (Segment 1)...
  日期 1215 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1215
    - Neurons: 16
    - Spikes: 226217

处理日期 1216 (Segment 2)...
  日期 1216 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1216
    - Neurons: 22
    - Spikes: 195411

处理日期 1217 (Segment 3)...
  日期 1217 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1217
    - Neurons: 19
    - Spikes: 276875

处理日期 1218 (Segment 4)...
  日期 1218 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1218
    - Neurons: 22
    - Spikes: 308436

处理日期 1219 (Segment 5)...
  日期 1219 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/cliqu

estimate_sparsity (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

...

compute_waveforms (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

计算channel_snr...
完成channel_snr计算，共处理29个units

处理日期 1214 (Segment 0)...
  日期 1214 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1214
    - Neurons: 13
    - Spikes: 155578

处理日期 1215 (Segment 1)...
  日期 1215 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1215
    - Neurons: 23
    - Spikes: 132386

处理日期 1216 (Segment 2)...
  日期 1216 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1216
    - Neurons: 22
    - Spikes: 151201

处理日期 1217 (Segment 3)...
  日期 1217 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1217
    - Neurons: 22
    - Spikes: 283664

处理日期 1218 (Segment 4)...
  日期 1218 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_2/date_1218
    - Neurons: 19
    - Spikes: 335225

处理日期 1219 (Segment 5)...
  日期 1219 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/cliqu

estimate_sparsity (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

compute_waveforms (workers: 20 processes):   0%|          | 0/10809 [00:00<?, ?it/s]

计算channel_snr...
完成channel_snr计算，共处理32个units

处理日期 1214 (Segment 0)...
  日期 1214 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1214
    - Neurons: 31
    - Spikes: 300910

处理日期 1215 (Segment 1)...
  日期 1215 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1215
    - Neurons: 24
    - Spikes: 185695

处理日期 1216 (Segment 2)...
  日期 1216 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1216
    - Neurons: 27
    - Spikes: 164470

处理日期 1217 (Segment 3)...
  日期 1217 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1217
    - Neurons: 16
    - Spikes: 151860

处理日期 1218 (Segment 4)...
  日期 1218 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1218
    - Neurons: 17
    - Spikes: 271308

处理日期 1219 (Segment 5)...
  日期 1219 结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/cliqu